# Exploring Emotional Profiles in 1.38M TMDB Movies

**Dataset:** [TMDB Movie VAD + Emotion Intensity Scores](https://www.kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores)
**Source:** [TMDB Movies Dataset](https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies) enriched with NRC lexicons by Saif M. Mohammad (NRC Canada)

Each movie is scored on 12 affective dimensions derived from plot keywords:
- **VAD** — valence (positive/negative), arousal (calm/excited), dominance (submissive/powerful)
- **8 emotions** — anger, anticipation, disgust, fear, joy, sadness, surprise, trust (0–1 intensity)

Scores represent *keyword associations*, not audience reaction. ~23% of movies have keyword coverage.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Resolve input path — works whether notebook is created from dataset page or pushed via CLI
from pathlib import Path
matches = sorted(Path('/kaggle/input').rglob('movie_vad_scores.csv'))
INPUT_PATH = matches[0]
print(f'Input: {INPUT_PATH}')

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'Rows: {len(df):,}   Columns: {len(df.columns)}')

## 1. Schema and coverage

In [ ]:
EMOTIONS = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']
VAD = ['valence', 'arousal', 'dominance']

print('Sentiment distribution:')
print(df['sentiment'].value_counts().to_string())
print(f'\nMovies with keyword coverage: {(df.sentiment != "unknown").sum():,} ({(df.sentiment != "unknown").mean():.1%})')
print('\nEmotion score means (scored movies only):')
scored = df[df['sentiment'] != 'unknown']
print(scored[EMOTIONS].mean().round(3).to_string())
print('\nVAD means:')
print(scored[VAD].mean().round(3).to_string())

## 2. Top films by emotion

In [ ]:
# Require at least 5 keywords for reliable signal
has_kw = df[df['keywords'].notna() & (df['keywords'].str.count(',') >= 4)]

print('Most joyful (by keyword associations):')
display(has_kw.nlargest(8, 'joy')[['title', 'genres', 'release_date', 'joy', 'sadness', 'sentiment']].reset_index(drop=True))

print('\nMost fear-associated:')
display(has_kw.nlargest(8, 'fear')[['title', 'genres', 'release_date', 'fear', 'anger', 'sentiment']].reset_index(drop=True))

print('\nMost negative valence:')
display(has_kw.nsmallest(8, 'valence')[['title', 'genres', 'release_date', 'valence', 'arousal', 'sentiment']].reset_index(drop=True))

## 3. Genre emotional fingerprints

In [ ]:
genre_df = (
    df[df['genres'].notna() & (df['valence'].notna())]
    .assign(genre=df['genres'].str.split(', '))
    .explode('genre')
)

FEATURES = VAD + EMOTIONS
genre_means = genre_df.groupby('genre')[FEATURES].mean()
genre_means['n_movies'] = genre_df.groupby('genre').size()
genre_means = genre_means[genre_means['n_movies'] >= 50].copy()

display(genre_means[VAD + ['n_movies']].sort_values('valence', ascending=False).round(3).head(12))

## 4. Valence × Arousal map by genre

Each point is a genre. Position = emotional tone (valence × arousal). Size = dominance.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

top_genres = genre_means.sort_values('n_movies', ascending=False).head(18)
colors = cm.tab20(np.linspace(0, 1, len(top_genres)))

for (genre, row), color in zip(top_genres.iterrows(), colors):
    size = 200 + row['dominance'] * 600
    ax.scatter(row['valence'], row['arousal'], s=size, color=color, alpha=0.85, zorder=3)
    ax.annotate(genre, (row['valence'], row['arousal']),
                fontsize=9, ha='center', va='bottom',
                xytext=(0, 7), textcoords='offset points')

ax.axvline(0.5, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axhline(0.5, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Valence  (0 = negative  →  1 = positive)', fontsize=11)
ax.set_ylabel('Arousal  (0 = calm  →  1 = excited)', fontsize=11)
ax.set_title('Genre Emotional Map — NRC VAD scores from TMDB keywords\n(dot size = dominance)', fontsize=12)
ax.set_xlim(0.3, 0.75); ax.set_ylim(0.3, 0.65)
ax.text(0.32, 0.31, 'Low energy, negative', fontsize=8, color='gray', alpha=0.7)
ax.text(0.60, 0.62, 'High energy, positive', fontsize=8, color='gray', alpha=0.7)
plt.tight_layout()
plt.show()

## Case study: The Sixth Sense & Alien

Why did these films score the way they did?

In [ ]:
sixth = df[df['title'].str.contains('Sixth Sense', case=False, na=False)].iloc[0]

print('Title:   ', sixth['title'])
print('Keywords:', sixth['keywords'])
print()
print('--- VAD scores ---')
for d in ['valence', 'arousal', 'dominance']:
    print(f'  {d:12s}: {sixth[d]:.3f}')
print()
print('--- Emotion scores ---')
for e in EMOTIONS:
    print(f'  {e:14s}: {sixth[e]:.3f}')
print()
print('--- Keyword groups (VAD valence buckets) ---')
kw_cols = ['positive_keywords', 'negative_keywords', 'unknown_keywords']
missing = [c for c in kw_cols if c not in df.columns]
if missing:
    print(f'  NOTE: columns not in this dataset version: {missing}')
else:
    print('  positive:', sixth['positive_keywords'])
    print('  negative:', sixth['negative_keywords'])
    print('  unknown: ', sixth['unknown_keywords'])
print()
print('Sentiment:', sixth['sentiment'])

# ── Alien ─────────────────────────────────────────────────────────────────
alien = df[df['title'].str.lower() == 'alien'].iloc[0]

print()
print('Title:   ', alien['title'])
print('Keywords:', alien['keywords'])
print()
print('--- VAD scores ---')
for d in ['valence', 'arousal', 'dominance']:
    print(f'  {d:12s}: {alien[d]:.3f}')
print()
print('--- Emotion scores ---')
for e in EMOTIONS:
    print(f'  {e:14s}: {alien[e]:.3f}')
print()
print('--- Keyword groups (VAD valence buckets) ---')
kw_cols = ['positive_keywords', 'negative_keywords', 'unknown_keywords']
missing = [c for c in kw_cols if c not in df.columns]
if missing:
    print(f'  NOTE: columns not in this dataset version: {missing}')
else:
    print('  positive:', alien['positive_keywords'])
    print('  negative:', alien['negative_keywords'])
    print('  unknown: ', alien['unknown_keywords'])
print()
print('Sentiment:', alien['sentiment'])

## Caveats

1. **Scores are relative, not absolute.** Valence 0.65 means "more positive-associated than 0.55" — not "65% positive."
2. **Keyword associations, not audience emotion.** A horror film with "fear" keywords scores high on fear — that's an association, not a review.
3. **~77% of movies have no keywords** and are scored `unknown`. Analyses above use the 23% with coverage.

See: Mohammad (2020), [Practical and Ethical Considerations in the Effective use of Emotion and Sentiment Lexicons](https://www.saifmohammad.com/WebDocs/EmoLex-Ethics-Data-Statement.pdf).